# FEWS farm model — Jones (2022), *Environment Systems and Decisions*

The stochastic programming model behind
[10.1007/s10669-021-09838-8](https://doi.org/10.1007/s10669-021-09838-8):
a farm choosing alternative water and electricity capacity under uncertain
precipitation, over 25 years and 4,000 Monte Carlo weather draws.

**This notebook is thin on purpose.** It imports the package and calls it; it
holds no model logic of its own. The model lives in `src/fews_stochopt/`, and
duplicating it here would create a second copy with nothing comparing the two —
which is the failure the repository's own tests exist to prevent.

## What runs without a licence

**Section 2 needs no solver at all.** The repository ships the frozen model
instances and their solutions, and verifying them is arithmetic. That is this
repository's central claim and it is the part that runs everywhere.

**Sections 3 and 4 need `gurobipy` but not a licence file.** The models solved
there are 37 variables; the licence that ships with `pip install gurobipy` allows
2,000. Only the full 704,002-variable formulation in section 5 needs a real
licence, and section 5 is optional.

## 1. Install

On Colab, **clone the repository and install from that checkout.**
`pip install git+https://…` is not enough and fails in a way that looks like a
bug in the model: it installs the package but not `data/`, because the
precipitation inputs live at the repository root and are not part of the wheel.
The installed `config.py` then derives the repository root relative to
site-packages and the first cell that loads data dies with a `FileNotFoundError`
naming a directory that has nothing to do with the problem.

In [1]:
import subprocess
import sys

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    REPO = "https://github.com/sear-labs/fews-stochopt-esd-2022.git"
    subprocess.run(["git", "clone", "--depth", "1", REPO], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e",
                    "fews-stochopt-esd-2022"], check=True)
    print("cloned and installed from the checkout")
else:
    print("running from a local checkout; `pip install -e .` once if you have not")

import fews_stochopt

print("fews_stochopt", fews_stochopt.__version__)

running from a local checkout; `pip install -e .` once if you have not


fews_stochopt 1.0.0


## 2. Verify the published result — no solver, no licence

Re-solving and checking are different verbs, and checking is the stronger one.

Given a model and a claimed solution, feasibility and the objective can be
confirmed by arithmetic. That verification needs no licence, does not depend on
hardware or solver version, and reproduces bit for bit forever — none of which is
true of re-solving.

`scripts/verify_solution.py` reads the twelve frozen instances in `artifacts/`
and checks each one. It imports nothing from this package: it writes the model
out again from the shipped instance files, because a check written from the same
source as the thing it checks is not a check.

In [2]:
import pathlib

import fews_stochopt

# Derived from the INSTALLED package, never from the working directory. A
# relative "../scripts/..." breaks the moment the notebook runs from anywhere but
# its own folder, which is exactly what happens on Colab after the clone.
ROOT = pathlib.Path(fews_stochopt.__file__).resolve().parents[2]


def run_script(name, *args):
    """Run a repository script and show BOTH streams and the return code.

    Printing only stdout is how a failing script comes to look like one that did
    nothing.
    """
    proc = subprocess.run(
        [sys.executable, str(ROOT / "scripts" / name), *args],
        capture_output=True, text=True, cwd=str(ROOT),
    )
    if proc.stdout:
        print(proc.stdout.rstrip())
    if proc.stderr:
        print("--- stderr ---", proc.stderr.rstrip(), sep="\n")
    if proc.returncode != 0:
        raise RuntimeError(f"{name} exited {proc.returncode}")
    return proc


run_script("verify_solution.py")

Verifying 12 model(s) from C:\Users\jonesec\dev\repo\projects\fews-stochopt-esd-2022\artifacts, with no solver and no licence.

  DML_expected_value
    5 weighted blocks, 4,000 runs, 35 variables   (first stage fixed)
    recomputed objective   1,898,261.671957
    claimed in .sol        1,898,261.671957   (delta 0.000e+00)
    worst row violation    0.000e+00
    worst bound violation  0.000e+00
    best independent search1,898,261.679250   (shipped value is +0.0073 from it)
    FEASIBLE, objective confirmed, and optimal to within $0.0073
  DML_known_climate_unknown_weather_climate1
    4 weighted blocks, 2,400 runs, 30 variables
    recomputed objective   1,648,352.523588
    claimed in .sol        1,648,352.523588   (delta 0.000e+00)
    worst row violation    0.000e+00
    worst bound violation  0.000e+00
    best independent search1,648,352.604741   (shipped value is +0.0812 from it)
    FEASIBLE, objective confirmed, and optimal to within $0.0812
  DML_known_climate_unknown_weat

CompletedProcess(args=['C:\\Users\\jonesec\\AppData\\Local\\anaconda3\\python.exe', 'C:\\Users\\jonesec\\dev\\repo\\projects\\fews-stochopt-esd-2022\\scripts\\verify_solution.py'], returncode=0, stdout='Verifying 12 model(s) from C:\\Users\\jonesec\\dev\\repo\\projects\\fews-stochopt-esd-2022\\artifacts, with no solver and no licence.\n\n  DML_expected_value\n    5 weighted blocks, 4,000 runs, 35 variables   (first stage fixed)\n    recomputed objective   1,898,261.671957\n    claimed in .sol        1,898,261.671957   (delta 0.000e+00)\n    worst row violation    0.000e+00\n    worst bound violation  0.000e+00\n    best independent search1,898,261.679250   (shipped value is +0.0073 from it)\n    FEASIBLE, objective confirmed, and optimal to within $0.0073\n  DML_known_climate_unknown_weather_climate1\n    4 weighted blocks, 2,400 runs, 30 variables\n    recomputed objective   1,648,352.523588\n    claimed in .sol        1,648,352.523588   (delta 0.000e+00)\n    worst row violation    0

That is the paper's result checked, end to end, with numpy.

Each line reports three things: that the point satisfies every constraint and
bound, that recomputing the objective from the variable values gives the number
the repository reports, and how far an independent search can improve on it.

The last figure is small and **positive by design**. Gurobi returns points a
little *outside* the feasible region while reporting `OPTIMAL`, so every shipped
solution is clipped back inside before it is stored. The reported values are
therefore valid lower bounds, and the true optimum sits just above them.

## 3. What was actually solved

The instances are small enough to read. Each is one weighted block per distinct
precipitation value — the model collapses to that because, given the capacities,
nothing couples one run-year to another and precipitation takes only five values.

That is why 704,002 variables become 37, and why this notebook needs no licence.

In [3]:
import json

import pandas as pd

instance = json.loads((ROOT / "artifacts" / "EP_stochastic.json").read_text())

blocks = pd.DataFrame(instance["blocks"])
blocks["share_of_run_years"] = blocks["run_years"] / blocks["run_years"].sum()
print(f"Equally Probable, Stochastic scenario — {instance['n_runs']:,} runs "
      f"x {instance['years']} years")
display(blocks)

print("\ncoefficients:")
for k, v in sorted(instance["coefficients"].items()):
    print(f"  {k:32} {v:>16,.6f}")

Equally Probable, Stochastic scenario — 4,000 runs x 25 years


,precip_cm,run_years,share_of_run_years
0,8.89,11230,0.11230
1,26.67,25141,0.25141
2,53.34,29777,0.29777
3,80.01,20022,0.20022
4,106.68,13830,0.13830



coefficients:
  alt_elc_kwh_per_kw                   1,944.000000
  alt_water_kwh_per_cm                33,021.412137
  cost_alt_elc_per_kw                  1,500.000000
  cost_alt_water_per_cm               28,950.375546
  cost_irrigation_water_per_cm         1,361.996063
  cost_utility_elc_per_kwh                 0.080000
  crop_price_per_tonne                   200.000000
  hectares                               200.000000
  irrigation_kwh_per_cm                5,283.425942
  max_irrigation_cm                        7.620000
  yield_a0                                -3.747000
  yield_a1                                 0.201400
  yield_a2                                -0.001400


## 4. Re-solve the model

Three of the four scenarios collapse and solve in milliseconds. The cell prints
the model size first, because that is the number that decides whether a reader
without a licence can run this at all.

**Perfect Information does not collapse** — every run chooses its own capacities,
so the runs share nothing to aggregate over. It is 4,000 separate solves of 178
variables each, about two minutes, and it is skipped by default. Set
`QUICK = False` to run it.

In [4]:
import gurobipy as gp

from fews_stochopt import collapsed, load_config
from fews_stochopt.model import EXPECTED_VALUE, KNOWN_CLIMATE, STOCHASTIC

cfg = load_config()

env = gp.Env(params={"OutputFlag": 0})
weights, n_runs = collapsed.rain_weights(cfg, "EP")
probe = collapsed.build(cfg, weights, n_runs, env)
print(f"collapsed model: {probe.NumVars} variables, {probe.NumConstrs} linear "
      f"and {probe.NumQConstrs} quadratic constraints")
print(f"free pip-installed Gurobi licence allows 2,000 variables -> "
      f"{'fits' if probe.NumVars <= 2000 else 'DOES NOT FIT'}")
del probe, env

collapsed model: 37 variables, 25 linear and 5 quadratic constraints
free pip-installed Gurobi licence allows 2,000 variables -> fits


In [5]:
QUICK = True   # skip the 4,000-solve Perfect Information scenario

rows = []
for site in cfg.sites:
    for scenario in (STOCHASTIC, KNOWN_CLIMATE, EXPECTED_VALUE):
        result = collapsed.solve(cfg, site, scenario)
        rows.append({"site": site, "label": cfg.site(site).label,
                     "scenario": scenario, "mean_profit": result["objective"],
                     "alt_water_cap_cm": result["alt_water_cap"],
                     "alt_elc_cap_kW": result["alt_elc_cap"]})

solved = pd.DataFrame(rows)
display(solved.style.format({"mean_profit": "{:,.4f}",
                             "alt_water_cap_cm": "{:.6f}",
                             "alt_elc_cap_kW": "{:.4f}"}))

if QUICK:
    print("\nQUICK = True: Perfect Information was NOT solved here.")
    print("It is 4,000 solves of 178 variables, about two minutes, and it fits")
    print("the free licence too. Its published value is read from results/ below.")

,site,label,scenario,mean_profit,alt_water_cap_cm,alt_elc_cap_kW
0,EP,Equally Probable,Stochastic,"2,246,937.9654",5.445671,94.4106
1,EP,Equally Probable,"Known Climate, Unknown Weather","2,345,267.1351",6.681297,124.1274
2,EP,Equally Probable,Expected Value,"2,246,937.7193",5.445671,94.3444
3,DML,Dry Most Likely,Stochastic,"1,899,221.2526",10.954934,206.7938
4,DML,Dry Most Likely,"Known Climate, Unknown Weather","1,964,087.5963",11.426694,208.8607
5,DML,Dry Most Likely,Expected Value,"1,898,261.6792",11.853902,222.0639



QUICK = True: Perfect Information was NOT solved here.
It is 4,000 solves of 178 variables, about two minutes, and it fits
the free licence too. Its published value is read from results/ below.


### Against the published figures

The comparison is to `reference/simstatstrad.csv`, which holds the paper's
Table 5 as the original pipeline produced it.

In [6]:
published = pd.read_csv(ROOT / "reference" / "simstatstrad.csv")
published = published.set_index(["Climate_Probability", "sim"])["profit_mean"]

check = solved.assign(
    published=[published[(r.label, r.scenario)] for r in solved.itertuples()],
)
check["difference"] = check["mean_profit"] - check["published"]

display(check[["label", "scenario", "published", "mean_profit", "difference"]]
        .style.format({"published": "{:,.4f}", "mean_profit": "{:,.4f}",
                       "difference": "{:+.4f}"}))

worst = check["difference"].abs().max()
print(f"\nlargest difference: ${worst:,.4f} on profits of order $2,000,000")
print("The Dry Most Likely expected-value row is the known one - its first stage")
print("solves a problem whose objective is flat. See docs/reproduction-notes.md.")

,label,scenario,published,mean_profit,difference
0,Equally Probable,Stochastic,"2,246,937.9615","2,246,937.9654",+0.0039
1,Equally Probable,"Known Climate, Unknown Weather","2,345,266.7851","2,345,267.1351",+0.3500
2,Equally Probable,Expected Value,"2,246,937.4728","2,246,937.7193",+0.2465
3,Dry Most Likely,Stochastic,"1,899,221.2314","1,899,221.2526",+0.0212
4,Dry Most Likely,"Known Climate, Unknown Weather","1,964,087.2089","1,964,087.5963",+0.3874
5,Dry Most Likely,Expected Value,"1,898,280.3314","1,898,261.6792",-18.6522



largest difference: $18.6522 on profits of order $2,000,000
The Dry Most Likely expected-value row is the known one - its first stage
solves a problem whose objective is flat. See docs/reproduction-notes.md.


## 5. The full pipeline — optional, and needs a real licence

Everything above used the collapsed model. `scripts/run_all.py` solves the
original 704,002-variable formulation as the published run did, reproduces both
of the paper's tables, and takes about eight minutes.

It needs a full Gurobi licence. On Colab a node-locked licence cannot work — the
virtual machine differs every session — so use WLS credentials held as Colab
secrets, never as literals in the notebook: a key committed to a repository is
exposed the moment the repository is shared, and deleting it later does not
remove it from the history.

```python
import os
from google.colab import userdata          # SecretNotFoundError is EXPECTED
for k in ("WLSACCESSID", "WLSSECRET", "LICENSEID"):
    os.environ["GRB_" + k] = userdata.get("GRB_" + k)
```

Then:

```bash
python scripts/run_all.py
```

`tests/test_collapsed_agrees.py` is what ties the two together: it solves the
collapsed model and the full one and asserts they agree, so the fast path above
is not a different model with a convenient answer.